In [9]:
!pip install python-dotenv

In [3]:
!pip install tuya-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 6.8 MB/s eta 0:00:0000:0100:01


In [49]:
from dotenv import load_dotenv
import os
import hashlib
import random
import string
import time
import requests
from tuya_connector import TuyaOpenAPI
from pprint import pp

def initialize_tuya_api():
    access_id = os.getenv("TUYA_ACCESS_ID")
    access_key = os.getenv("TUYA_ACCESS_KEY")
    if not all([access_id, access_key]):
        raise ValueError("Tuya credentials must be set as environment variables.")
    openapi = TuyaOpenAPI("https://openapi.tuyaus.com", access_id, access_key)
    openapi.connect()
    return openapi
    
def send_tuya_command(openapi, endpoint, commands):
    response = openapi.post(endpoint, commands)
    if response.get("success"):
        print("Command executed successfully")
    else:
        print(f"Failed to execute command: {response.get('msg')}")
        
def calculate_sha512(text):
    return hashlib.sha512(text.encode('utf-8')).hexdigest()

def add_authorization_parameters(method, parameters, key, secret):
    parameters["apiKey"] = key
    parameters["time"] = str(int(time.time()))

    rand = ''.join(random.choices(string.ascii_lowercase, k=6))
    
    sorted_params = sorted(parameters.items())
    query_string = '&'.join(f'{k}={v}' for k, v in sorted_params)
    
    api_sig_base = f"{rand}/{method}?{query_string}#{secret}"
    api_sig = rand + calculate_sha512(api_sig_base)
    
    parameters["apiSig"] = api_sig

# Load environment variables from a specified .env file
load_dotenv(dotenv_path='/home/jovyan/work/.env')

def codeforces_api_request(method, parameters):
    base_url = "https://codeforces.com/api/"
    url = f"{base_url}{method}"
    if not parameters:
        parameters = {}
    key = os.getenv("CODEFORCES_API_KEY")
    secret = os.getenv("CODEFORCES_API_SECRET")
    if not key or not secret:
        raise ValueError("API key and secret must be set as environment variables.")
    
    add_authorization_parameters(method, parameters, key, secret)
    
    combined_parameters = '&'.join(f'{k}={v}' for k, v in parameters.items())
    full_url = f"{url}?{combined_parameters}"
    pp(f"Parameters before :: {parameters}")  # For debugging
    pp(f"New Combined Parameters are :: {combined_parameters}")  # For debugging
    pp(f"Formed Request is as follows :: {full_url}")  # For debugging
    response = requests.get(full_url)
    if response.status_code == 200:
        return response.json()
    else:
        print("Failed to fetch data")
        return None
        
def map_rating_to_color(rating):
    # Map Codeforces rating to HSV color
    if rating < 1200:
        return {"h": 240, "s": 1000, "v": 1000}  # Blue
    elif rating < 1400:
        return {"h": 120, "s": 1000, "v": 1000}  # Green
    elif rating < 1600:
        return {"h": 180, "s": 1000, "v": 1000}  # Cyan
    elif rating < 1900:
        return {"h": 60, "s": 1000, "v": 1000}   # Yellow
    elif rating < 2100:
        return {"h": 30, "s": 1000, "v": 1000}   # Orange
    else:
        return {"h": 0, "s": 1000, "v": 1000}    # Red
        
def set_bulb_color(openapi, color):
    bulb_id = os.getenv("TUYA_BULB_ID")
    if not bulb_id:
        raise ValueError("Tuya bulb ID must be set as an environment variable.")
    
    commands = {
        "commands": [
            {"code": "switch_led", "value": True},
            {"code": "bright_value_v2", "value": 1000},
            {"code": "colour_data_v2", "value": color}
        ]
    }
    
    send_tuya_command(openapi, f"/v1.0/iot-03/devices/{bulb_id}/commands", commands)
    
def contest_status(contestId, handle, asManager=False, submissionReturn=None, count=None):
    if not contestId:
        raise ValueError("ContestId must be provided")
    contestId = str(contestId)
    method = "contest.status"
    parameters = {}
    parameters["contestId"] = contestId
    if handle:
        parameters["handle"] = handle
    if asManager:
        parameters["asManager"] = asManager
    if submissionReturn:
        parameters["from"] = submissionReturn
    if count:
        parameters["count"] = count
    data = codeforces_api_request(method, parameters)
    return data
    
def contest_list():
    method = "contest.list"
    return codeforces_api_request(method, {})


def main():
    # pp(contest_status(1985, "hanisntsolo"))
    pp(contest_list())
    # method = "user.status"
    # parameters = {"handle": handle}
    #  https://codeforces.com/api/contest.status?contestId=566&asManager=true&from=1&count=10
    # data = codeforces_api_request(method, parameters)
    # pp(data)
    pp("#########################################################################")
    handle = "hanisntsolo"
    method = "user.info"
    parameters = {"handles": handle}
    data = codeforces_api_request(method)
    print(data)
    if data:
        user_info = data['result'][0]
        rating = user_info['rating']
        color = map_rating_to_color(rating)
        openapi = initialize_tuya_api()
        set_bulb_color(openapi, color)
    
if __name__ == "__main__":
    main()


("Parameters before :: {'apiKey': '4a830dd3bb455948e4e180f0b2db38b61013caa1', "
 "'time': '1720210941', 'apiSig': "
 "'eqrokx6f29d254e1642e8ae220dc0fda5490402b512303e616b2c2d32a57ef939e65cc0fd73afa6927d81a7304d43a1b08c20b0d52e6fbc907d933c67d2f790040f8cc'}")
('New Combined Parameters are :: '
 'apiKey=4a830dd3bb455948e4e180f0b2db38b61013caa1&time=1720210941&apiSig=eqrokx6f29d254e1642e8ae220dc0fda5490402b512303e616b2c2d32a57ef939e65cc0fd73afa6927d81a7304d43a1b08c20b0d52e6fbc907d933c67d2f790040f8cc')
('Formed Request is as follows :: '
 'https://codeforces.com/api/contest.list?apiKey=4a830dd3bb455948e4e180f0b2db38b61013caa1&time=1720210941&apiSig=eqrokx6f29d254e1642e8ae220dc0fda5490402b512303e616b2c2d32a57ef939e65cc0fd73afa6927d81a7304d43a1b08c20b0d52e6fbc907d933c67d2f790040f8cc')
{'status': 'OK',
 'result': [{'id': 1991,
             'name': 'Codeforces Round (Div. 1 + Div. 2)',
             'type': 'CF',
             'phase': 'BEFORE',
             'frozen': False,
             'duratio

TypeError: codeforces_api_request() missing 1 required positional argument: 'parameters'